# Export and flatten a Plot3D mesh with plot3d-flatten

This tutorial explains -- in detail -- the plot3d-flatten deck format (a
Plot3D grid + connectivity.json + boundary_conditions.yaml bundle meant for
GPU CFD solvers), and walks through generating every piece of it from a
Plot3D multi-block mesh using `plot3d.glennht`.

This is not just an API-call tour. The goal is that by the end you understand
**exactly** what is inside a `*.connectivity.json` file, why each field is
there, and how the pieces fit together, so you can produce these files for
your own meshes (or debug one that doesn't load).

## The big picture: three inputs, two producers

A GPU CFD solver consuming the plot3d-flatten deck format loads a case from
**three** artifacts:

| # | File | What it is | Who produces it |
|---|------|------------|------------------|
| (a) | `{case}.xyz` | A formatted (ASCII) Plot3D multi-block grid -- the raw node coordinates. | `plot3d.write_plot3D` |
| (b) | `{case}.connectivity.json` | The **block-level graph**: which block faces touch which other block faces, which faces are exterior boundaries (and what they physically are), and which faces are rotationally periodic. | `plot3d.glennht.write_connectivity_json` (this tutorial) |
| (c) | a run YAML's `boundary_conditions:` list | The **physics** attached to each boundary surface id -- inlet total conditions, outlet back pressure, wall thermal condition, etc. | `plot3d.glennht.write_boundary_conditions_yaml` (a fragment you paste/merge into your run deck) |

The important thing to internalize: the GPU solver builds its own internal
**flat, per-cell dual graph** (the thing it actually walks during a
timestep) *itself*, at load time, from the grid + connectivity.json. We never
compute or ship a cell-level graph from Python -- we only supply the much
smaller **block-level** graph (which whole faces of which whole blocks touch),
and the solver expands that into per-cell adjacency internally. That's what
keeps `connectivity.json` small (a few hundred lines) even for meshes with
millions of cells.

This package (`plot3d`) already computes block connectivity, exterior
("outer") faces, and rotational periodicity for Plot3D meshes -- that's the
whole reason `plot3d.connectivity` and `plot3d.periodicity` exist. The
`plot3d.glennht` module in this tutorial is a thin translation layer: it takes
those existing Python data structures and serializes them into the
plot3d-flatten deck's JSON/YAML shapes, instead of round-tripping through
upstream mesh-preprocessing tooling.

## Install

The graph-export feature (`plot3d.glennht.plot3d_flatten_deck` / `plot3d.glennht.plot3d_flatten_bc`)
requires **plot3d 1.10.0 or newer** from PyPI:

```
!pip install "plot3d>=1.10.0"
```

Note: this tutorial does **not** need `pymetis`/METIS. That's a separate,
optional dependency used for partitioning very large meshes across multiple
CPUs/GPUs -- unrelated to graph export.

In [ ]:
!pip install "plot3d>=1.10.0"

## Imports

Everything used in this tutorial is importable from the top-level `plot3d`
package or from `plot3d.glennht`.

In [ ]:
import json
import math
import os
import pprint

import numpy as np

from plot3d import Block
from plot3d.connectivity import connectivity, PERMUTATION_MATRICES
from plot3d.periodicity import rotated_periodicity, create_rotation_matrix
from plot3d.glennht import (
    write_connectivity_json,
    tag_surfaces_from_diagonals,
    tag_surfaces_from_bc_codes,
    tag_surfaces_geometric,
    write_bc_codes_json,
    merge_connectivity_json,
    write_boundary_conditions_yaml,
    Plot3DFlattenInletBC,
    Plot3DFlattenOutletBC,
    Plot3DFlattenWallBC,
    export_to_plot3d_flatten_deck,
)

pp = pprint.PrettyPrinter(indent=2, width=100)
OUT_DIR = "wedge_out"
os.makedirs(OUT_DIR, exist_ok=True)

## The `connectivity.json` schema, field by field

Before generating one, here is every top-level key that the
plot3d-flatten deck format understands:

| Field | Type | Meaning |
|---|---|---|
| `mesh_file` | string | The paired `.xyz` grid's filename (informational/self-describing -- the run YAML's `mesh.grid_file` is what the solver actually opens). |
| `nblocks` | int | Number of blocks in the mesh. Must match the block count in `mesh_file`. |
| `face_matches` | list of face-match records | Interior, block-to-block conformal interfaces: whole faces of block A that are node-for-node coincident with a whole (or partial) face of block B. |
| `outer_faces` | list of outer-face records | Faces with **no** interior match -- i.e. the exterior boundary of the domain. Each carries an integer `id` identifying *which physical boundary surface* it belongs to (inlet, wall, ...). |
| `periodic_faces` | list of face-match records | Same shape as `face_matches`, but for pairs of faces that coincide only after a **rotation** by the blade pitch angle (rotationally-periodic seams), rather than being directly coincident. |
| `periodicity` | dict | Global periodicity metadata, required whenever `periodic_faces` is non-empty (see below). |
| `surface_ids` | dict `{"<id>": name}` | Human-readable name for every integer surface id used in `outer_faces[].id`. The run YAML's `boundary_conditions:` entries reference these same integer ids in their `surfaces:` list. |
| `permutation_matrices` | list of 8 2x2 matrices | A fixed reference copy of the 8 canonical orientation matrices (see below) -- always the same 8 matrices, included so the file (and any independent parser) is self-describing. |

The `periodicity` dict itself has these sub-fields:

| Field | Type | Meaning |
|---|---|---|
| `transformation_matrix` | 3x3 matrix | Rotation matrix such that `Face_B_points = (transformation_matrix @ Face_A_points.T).T` maps one periodic face onto its partner. |
| `rotation_angle_rad` / `rotation_angle_deg` | float | The blade pitch angle (`2*pi / nblades`), in both units for convenience. |
| `nblades` | int | Blade count for a full annulus (used to scale flow between the periodic wedge and the full machine). |
| `rotation_axis` | `"x"`\|`"y"`\|`"z"` | The machine's axis of rotation. |
| `convention` | string | States, in words, the direction the `transformation_matrix` maps (A -> B), so there's no ambiguity about which way to apply it. |
| `source` | string | Provenance -- which tool/function produced this file. |

### The diagonal (`lb`/`ub`) convention

Every face -- whether an interior match, an outer face, or a periodic pair --
is stored as **two 3-integer node-index corners**, `lb = [i, j, k]` and
`ub = [i, j, k]`, both **0-based**. A structured face is always a planar
subset of one block's node grid, so exactly one of the three axes is
*collapsed*: `lb[axis] == ub[axis]`. That constant axis and its value tell you
**which of the 6 logical block faces** this is:

- axis 0 (`i`) constant at `0` -> the block's `I=1` face; constant at `IMAX-1` -> `I=IMAX`.
- axis 1 (`j`) constant -> `J=1` or `J=JMAX`.
- axis 2 (`k`) constant -> `K=1` or `K=KMAX`.

The other two axes vary from their `lb` to `ub` value and describe the face's
in-plane extent (a sub-region of the full block face, if the match is
partial).

For `face_matches` and `periodic_faces`, **block1**'s `lb`/`ub` is always
given in ascending (min/max) order, but **block2**'s corners are *directional*:
they encode the traversal order needed to walk block2's face in lock-step with
block1's face. Concretely, this means block2's `lb`/`ub` pair can have
`il > ih` (or similarly for j/k) on the *varying* axes -- that's not a typo,
it's how a reversed axis (block2's index decreases while block1's increases)
is represented without a separate "reversed" flag. You'll see exactly this in
the schema dissection below once we've generated a real file.

### `face_matches` vs `outer_faces` vs `periodic_faces`

These three lists partition every face of every block into three physically
different roles:

- **`face_matches`** -- interior seams. Both sides are real, matched blocks;
  flux/state gets exchanged directly across the interface. This is what
  `plot3d.connectivity.connectivity` finds by comparing every block's
  exterior faces against every other block's.
- **`outer_faces`** -- true exterior boundaries: faces with no matching
  neighbor at all. These need a physical boundary condition (inlet, outlet,
  wall, ...), which is why each one carries an integer `id` -- see "surface
  ids and tagging" below.
- **`periodic_faces`** -- a special case of "interior seam", but the two
  sides don't coincide directly; they coincide *after rotating one of them by
  the blade pitch angle*. These start out looking like outer faces (nothing
  matched them in the plain `connectivity()` pass) and are only reclassified
  once `plot3d.periodicity.rotated_periodicity` checks them against a
  rotated copy of the mesh.

Every record in `face_matches` and `periodic_faces` has the same shape:

```json
{
  "block1": {"block_index": 0, "lb": [i, j, k], "ub": [i, j, k]},
  "block2": {"block_index": 1, "lb": [i, j, k], "ub": [i, j, k]},
  "permutation_index": -1,
  "permutation_matrix": [[1, 0], [0, 1]]
}
```

while every record in `outer_faces` additionally carries the surface `id`:

```json
{"block_index": 0, "lb": [i, j, k], "ub": [i, j, k], "id": 1}
```

### `permutation_index` and `permutation_matrix`

When two faces match, their local in-plane `(u, v)` parametric axes might
not line up directly -- one or both could be reversed, and/or `u`/`v` could
be swapped (a "cross-plane" match, e.g. block1's face varies in `(j, k)` but
block2's varies in `(i, k)`). There are exactly `2 x 2 x 2 = 8` distinct
orientations, and `plot3d.connectivity.PERMUTATION_MATRICES` is the fixed
array of their 2x2 signed matrices:

```python
PERMUTATION_MATRICES = [
    [[ 1,  0], [ 0,  1]],   # 0: identity
    [[-1,  0], [ 0,  1]],   # 1: u reversed
    [[ 1,  0], [ 0, -1]],   # 2: v reversed
    [[-1,  0], [ 0, -1]],   # 3: both reversed
    [[ 0,  1], [ 1,  0]],   # 4: swapped
    [[ 0, -1], [ 1,  0]],   # 5: swap + u reversed
    [[ 0,  1], [-1,  0]],   # 6: swap + v reversed
    [[ 0, -1], [-1,  0]],   # 7: swap + both reversed
]
```

`write_connectivity_json` always writes **both** fields on every
`face_matches`/`periodic_faces` entry:

- `permutation_index`: for **in-plane** matches (indices 0-3), this is
  written as **`-1`**, because the traversal direction is already fully
  encoded in block2's directional `lb`/`ub` (see above) -- there's nothing
  extra to say. For **cross-plane** matches (indices 4-7), the real `0..7`
  index is written, because a simple `lb`/`ub` diagonal can't represent an
  axis swap.
- `permutation_matrix`: always the actual 2x2 matrix, regardless of the
  above.

This matters because the plot3d-flatten reader derives orientation from the
permutation matrix first: it derives the canonical orientation directly
from `permutation_matrix` whenever it's present, and only falls back to
`permutation_index.clamp(0, 7)` if the matrix is missing. A bare `-1` with
no matrix would silently clamp to `0` (identity) -- wrong for permutations
1-3. Writing both fields side-steps that trap entirely.

Connectivity JSON produced by upstream mesh-preprocessing tooling that emits
the plot3d-flatten deck format (rather than this Python exporter) carries only a flat
`permutation_index` with real small values instead of `-1`. For example, a
typical set of index values across several face matches might look like:

```
[0, 0, 0, 1, 0, 0, 0, 1, 1, 0]
```

Index `1` there is "u reversed" -- an ordinary case of two blocks meeting
face-to-face with one of them numbered in the opposite direction along that
edge. Our synthetic tutorial mesh below happens to produce only identity
matches (both blocks were generated with the same nested loops), so keep
this in mind as an example of a non-trivial orientation you may encounter
in real meshes.

### Surface ids and the three tagging strategies

An `outer_faces` entry's `id` is just an integer; its meaning comes from the
`surface_ids` map, e.g.:

```json
{"1": "inlet", "2": "outlet", "3": "blade", "4": "hub", "5": "shroud"}
```

These are exactly the ids the run YAML's `boundary_conditions:` entries
reference in their own `surfaces:` list (e.g. `surfaces: [3]` for a blade
wall BC). Nothing assigns these ids automatically -- `connectivity()` leaves
every outer face's `id` as a meaningless sequential counter (`1, 2, 3, ...`
in face-discovery order, with no physical significance whatsoever). You must
run one of three **tagging** functions to overwrite it with a real surface
id before writing `connectivity.json`:

| Strategy | Function | Use when |
|---|---|---|
| **Geometric** | `tag_surfaces_geometric` | Clean annular turbomachinery: classifies each outer face by its mean axial position and radius relative to the whole mesh's axial/radial extent (min-x band -> inlet, max-x band -> outlet, min-r -> hub, max-r -> shroud, everything else -> blade). Fast, no extra input needed, but assumes the machine's axis-of-rotation convention holds. |
| **BC codes** | `tag_surfaces_from_bc_codes` | You already have authoritative per-block-face integer codes (e.g. from a code-tagged legacy mesh). Exact, but requires that side information to exist. |
| **Diagonals** | `tag_surfaces_from_diagonals` | You know precisely, by hand, which `[i,j,k]` corner box on which block is which surface. Fully explicit, most tedious, always correct. |

All three mutate the `outer_faces` list in place (setting `id`) and return
`(outer_faces, surface_ids)`.

### What about multiple inlets or outlets?

Not a problem. Surfaces are just keyed by integer id, and each boundary
condition attaches to whichever ids it lists in `surfaces: [...]`. You can
have as many inlets and outlets as you want -- each one is its own surface
id, with its own physics.

There are two patterns for handling them:

1. **Distinct inlets/outlets with different conditions** (e.g. a primary
   inlet and a secondary cooling-air inlet at different total pressure and
   temperature): give each its own distinct surface id, and its own BC
   entry.
2. **Several faces that share identical conditions** (e.g. several outer
   faces that are all part of the same physical inlet plenum): tag them
   all with the *same* id, and list that one id once -- or, if they already
   ended up with different ids, just list them all in one BC entry's
   `surfaces:` list.

```python
# Pattern 1: two inlets, two different sets of conditions
inlet_main = Plot3DFlattenInletBC(
    name="main_inlet", surfaces=[1],
    total_pressure=350000.0, total_temperature=450.0,
)
inlet_cooling = Plot3DFlattenInletBC(
    name="cooling_inlet", surfaces=[6],
    total_pressure=360000.0, total_temperature=320.0,
)

# Pattern 2: several faces, one shared set of conditions
inlet_shared = Plot3DFlattenInletBC(
    name="m_inlet", surfaces=[1, 7, 9],
    total_pressure=101325.0, total_temperature=288.15,
)
outlet_shared = Plot3DFlattenOutletBC(
    name="m_outlet", surfaces=[2, 8],
    back_pressure=90000.0,
)
```

**Tagging caveat**: which tagging strategy you use matters here.
`tag_surfaces_from_bc_codes` and `tag_surfaces_geometric` map a *role* to a
single id -- every inlet-coded (or inlet-positioned) face becomes the same
one inlet surface. That's fine when all of those faces share the same
physics (Pattern 2 above), but it can't separate two inlets that need
*different* conditions, since they'd collapse to the same id.
`tag_surfaces_from_diagonals` lets you assign arbitrary, distinct ids/names
face by face, so reach for it whenever you have two physically-different
inlets (or outlets) that need to stay separate (Pattern 1 above).

#### A concrete multi-inlet / multi-outlet example

A single inlet and a single outlet, like the wedge built later in this
tutorial, is the simplest case -- but real machines usually aren't that
simple. A cooled turbine stage might have a main gas-path inlet *and* a
separate coolant-feed inlet, and split its exit flow into a core outlet and
a bypass/purge outlet. A multistage machine has its own inlet and outlet on
every row (see the multi-row merge section below). None of this changes the
mechanics: every inlet/outlet is still just a `Plot3DFlattenInletBC`/`Plot3DFlattenOutletBC`
pointing at whichever `surfaces: [...]` id(s) it owns. Here's a
four-boundary example -- two inlets, two outlets, three walls -- built and
dumped to the same YAML fragment a plot3d-flatten deck consumer reads.

In [ ]:
# A cooled turbine stage with TWO inlets (main gas path + a coolant feed)
# and TWO outlets (core exit + a bypass/purge exit). Each `surfaces` id
# here would come from tagging -- reach for tag_surfaces_from_diagonals to
# give each inlet/outlet its own distinct id, since two physically-different
# inlets can't share one id (see the tagging caveat just above).
multi_io_bcs = [
    Plot3DFlattenInletBC(
        name="main_inlet", surfaces=[1],
        total_pressure=101325.0, total_temperature=288.0,
        turbulence_intensity=0.05, turbulence_length_scale=1e-3,
    ),
    Plot3DFlattenInletBC(
        name="coolant_inlet", surfaces=[6],
        total_pressure=250000.0, total_temperature=320.0,
        turbulence_intensity=0.05, turbulence_length_scale=1e-3,
    ),
    Plot3DFlattenOutletBC(name="core_outlet", surfaces=[2], back_pressure=140000.0),
    Plot3DFlattenOutletBC(name="bypass_outlet", surfaces=[7], back_pressure=120000.0),
    Plot3DFlattenWallBC(name="blade", surfaces=[3], thermal="adiabatic"),
    Plot3DFlattenWallBC(name="hub", surfaces=[4], thermal="adiabatic"),
    Plot3DFlattenWallBC(name="shroud", surfaces=[5], thermal="adiabatic"),
]

multi_io_path = os.path.join(OUT_DIR, "multi_io_boundary_conditions.yaml")
multi_io_yaml = write_boundary_conditions_yaml(multi_io_bcs, multi_io_path)
print(multi_io_yaml)

Restating the two patterns from above, now visible in the printed YAML:
`main_inlet`/`coolant_inlet` and `core_outlet`/`bypass_outlet` are **Pattern
1** -- distinct ids, one BC each, different physics per boundary. If two of
those faces instead shared identical conditions, **Pattern 2** would apply:
list every shared id in a single BC's `surfaces:`, e.g.
`Plot3DFlattenInletBC(name="inlet", surfaces=[1, 6], ...)`.

Multistage/mixing-plane machines are the most common real-world source of
many inlets and outlets: each row contributes its own inlet and outlet, and
the tutorial's `merge_connectivity_json(..., id_stride=100)` (Step 8, below)
is exactly what turns row 1's local ids `1..5` into `101..105`, row 2's into
`201..205`, and so on. An interior row-to-row seam is then just an outlet BC
and an inlet BC that name each other via `mixing_plane_partner`, as shown in
the `boundary_conditions:` excerpt earlier in this section.

### The `bc_codes.json` sidecar

`write_bc_codes_json` writes a small companion file recording the *raw* per-block,
per-face integer face codes that a `bc_codes` tagging decision was based
on -- useful as an audit trail, and as the input format `tag_surfaces_from_bc_codes`
itself expects. One row of exactly 6 integers per block, in a fixed face
order:

```
FACE_ORDER = ["I=1", "I=IMAX", "J=1", "J=JMAX", "K=1", "K=KMAX"]
```

with the legend:

| Code | Meaning |
|---|---|
| 5 | inlet |
| 6 | outlet |
| 10 | blade |
| 9 | hub |
| 109 | shroud |
| 13 | periodic (low side) |
| 14 | periodic (high side) |
| 0 | interior interface |

Here's an illustrative excerpt of a `bc_codes.json` (5-block stator row) to
ground this in a realistic case:

```json
{
  "face_order": ["I=1", "I=IMAX", "J=1", "J=JMAX", "K=1", "K=KMAX"],
  "blocks": [
    [5, 6, 0, 14, 9, 109],
    [5, 6, 13, 0, 9, 109],
    [0, 0, 10, 0, 9, 109],
    ...
  ],
  "block_order": ["S35#0001", "S35#0002", ...]
}
```

Code `0` (interior interface) on a face that `tag_surfaces_from_bc_codes`
can't find among the current `outer_faces` (because `connectivity()` already
matched it) is simply a no-op. Code `0` on a face that *is* still an outer
face (i.e. `connectivity()` failed to block-match it) maps to the special id
**8**, `unmatched_interface` -- a CONFORMAL interior interface that the
Python connectivity search missed, *not* a wall; it's flagged distinctly so
you notice and fix the mesh/matching rather than silently treating a real
flow passage as a solid wall.

Codes `13`/`14` similarly only matter as a **safety net**: the intended
order of operations is to run `rotated_periodicity` *first* (it correctly
splits true periodic seams out of `outer_faces` into `periodic_faces`), and
only then tag whatever remains with `bc_codes`. Any face still coded
`13`/`14` at that point is one periodicity detection *missed* -- it gets
tagged id **6**, `periodic`, as a distinct "this looks periodic but wasn't
matched" bucket, instead of silently becoming a wall or blade surface.

### The `boundary_conditions:` YAML -- physics, not topology

`connectivity.json` only says *where* the boundaries are (as integer surface
ids). The actual physics goes in the run YAML's `boundary_conditions:` list,
each entry keyed by `type` and linked back to `connectivity.json` purely
through its `surfaces: [ids]` list. `plot3d.glennht.plot3d_flatten_bc`
gives you three small dataclasses for this, matching typical plot3d-flatten
deck run configurations:

- **`Plot3DFlattenInletBC`**: `total_pressure`, `total_temperature`, plus optional
  `flow_angle_deg`, `turbulence_intensity`, `turbulence_length_scale`,
  `blades_full_ring` (blade count for pitch scaling), and mixing-plane
  fields `inlet_subtype="mixing_plane"` / `mixing_plane_partner`.
- **`Plot3DFlattenOutletBC`**: `back_pressure`, plus optional `extrapolation_order`,
  `mixing_plane_partner`, `blades_full_ring`.
- **`Plot3DFlattenWallBC`**: `thermal` (`"adiabatic"` by default, or set
  `wall_temperature`/`wall_heat_flux` for isothermal/fixed-flux), plus
  `rotating` for walls that spin with their row's frame.

Every dataclass has a `surfaces: List[int]` field -- the `connectivity.json`
surface id(s) it applies to -- and only fields you actually set get emitted
(`to_dict()` skips anything left `None`), so a plain wall stays a 3-4 line
mapping. `write_boundary_conditions_yaml(bcs, filename, rotation=...)`
serializes a list of these to a YAML fragment, optionally alongside a
top-level `rotation:` block (per-block angular velocity for rotating rows).

For a **multi-row** machine (rotor + stator), a mixing-plane interface pairs
one row's outlet with the next row's inlet by name
(`mixing_plane_partner`), and the *same* physical seam gets scaled by each
row's own `blades_full_ring` (46 for a 46-blade stator, 36 for a 36-blade
rotor, say) so flow conserves correctly across the pitch change. Excerpt
from an example plot3d-flatten deck run configuration:

```yaml
boundary_conditions:
  - name: rotor35_out
    type: outlet
    surfaces: [2]
    back_pressure: 140293.8
    mixing_plane_partner: stator35_in
    blades_full_ring: 36

  - name: stator35_in
    type: inlet
    surfaces: [101]
    inlet_subtype: mixing_plane
    mixing_plane_partner: rotor35_out
    total_pressure: 140293.8
    total_temperature: 359.0
    blades_full_ring: 46
```

Notice the stator's inlet surface id is **101**, not 1 -- which brings us to
multi-row merging.

### Multi-row merge: why ids get a per-row offset

A multistage machine (rotor + stator, or more rows) is typically meshed and
connectivity-analyzed **one row at a time** -- each row's blocks are
generated and its own `connectivity()` / `rotated_periodicity()` pass is run
independently, producing a `{row}.connectivity.json` per row. Those per-row
grids are then concatenated into one `.xyz` (rotor blocks first, then
stator blocks, in flow order), and `merge_connectivity_json` combines the
per-row JSON payloads to match:

- every `block_index` in `face_matches`/`outer_faces`/`periodic_faces` is
  shifted by the running total of blocks from prior rows, so block indices
  stay unique and in sync with the concatenated grid;
- every `outer_faces[].id` is shifted by `id_stride * row` (default
  `id_stride=100`) -- so row 0 (rotor) keeps ids `1..5`, row 1 (stator)
  becomes `101..105`, row 2 would be `201..205`, etc. That's exactly the
  `surfaces: [101]` you saw above;
- each row's own `periodic_faces` entries get stamped with *that row's own*
  `rotation_angle_rad` (its blade pitch), since different rows can have
  different blade counts and hence different pitch angles;
- the top-level `periodicity` block in the merged file is just row 0's, as
  a fallback default -- the authoritative per-seam pitch angle travels on
  each `periodic_faces` entry itself.

The interior `face_matches` connecting one row to the next (the mixing-plane
seam) are **not** produced by this merge step -- they still need their own
connectivity/periodicity pass if the rows are meshed to be conformal at the
seam, or are simply left as two independent `outer_faces` (mixing-plane
inlet/outlet pair) if they are not conformal, exactly as in the example run
configuration above (surfaces `2` and `101` are two separate outer faces,
coupled only through the solver's mixing-plane BC, not through a
`face_matches` entry).

## A runnable example: a two-block annular wedge

To make every step above concrete -- and fast enough to run on Colab's free
tier in a couple of seconds -- we'll build a tiny **annular wedge**: two
blocks stacked along `x` (like a 1.5-block rotor/stator stub), each spanning
a fixed hub-to-shroud radius range and a small wedge of angle `dtheta` in
`theta`. This single-passage geometry naturally produces exactly the face
types we want to demonstrate:

- an **interior match** where the two blocks meet (shared `x` plane),
- **inlet**/**outlet** outer faces at the two open `x` ends,
- **hub**/**shroud** outer faces at min/max radius,
- a **rotationally-periodic** pair on each block's two `theta` faces (since
  `dtheta` is an exact submultiple of `2*pi`).

With `dtheta = 20 deg`, `nblades = 360/20 = 18` -- i.e. this wedge is a
1/18th slice of a full annulus.

In [ ]:
def wedge_block(x0, x1, nx, r_hub=0.2, r_shroud=0.3, nr=9, dtheta=np.radians(20.0), nt=13):
    x = np.linspace(x0, x1, nx)
    r = np.linspace(r_hub, r_shroud, nr)
    th = np.linspace(0.0, dtheta, nt)
    X = np.zeros((nx, nr, nt))
    Y = np.zeros_like(X)
    Z = np.zeros_like(X)
    for i in range(nx):
        for j in range(nr):
            for k in range(nt):
                X[i, j, k] = x[i]
                Y[i, j, k] = r[j] * np.cos(th[k])
                Z[i, j, k] = r[j] * np.sin(th[k])
    return Block(X, Y, Z)

dtheta = np.radians(20.0)
nblades = round(2 * np.pi / dtheta)  # 18

# Two blocks stacked along x, sharing the x=0.5 face
blocks = [wedge_block(0.0, 0.5, 15), wedge_block(0.5, 1.0, 15)]

print(f"nblades = {nblades}")
print("blocks:", [(b.IMAX, b.JMAX, b.KMAX) for b in blocks])

### Visualizing the mesh

Before computing any connectivity, let's *see* what we just built. This is an **annular wedge** -- a single blade passage -- split into **2 blocks** stacked along the axial (`x`) direction. `plot_blocks` draws each block's grid lines and nodes in a different color, so the two colors below are exactly the two entries of `blocks`.

In [ ]:
from plot3d import plot_blocks

plot_blocks(blocks)

## Step 1: Full connectivity

We use `plot3d.connectivity.connectivity`, **not**
`plot3d.connectivity.connectivity_fast`. `connectivity_fast` first reduces
every block by their common GCD to speed up matching -- a great trick for
huge, regular meshes, but that reduction can silently *drop* real matches on
O-grids or H-split topologies where the reduced index spacing no longer
lines up. This isn't theoretical: on a real turbomachinery stator mesh, the
fast GCD-reduced path has been observed to drop half of the real face
matches, mis-tagging them as unmatched exterior interfaces. For graph
export -- where a missed interior match becomes a phantom wall the solver
will (wrongly) apply a no-slip condition to -- always use the full
`connectivity()`.

`connectivity()` returns `(face_matches, outer_faces)`. Our two blocks meet
at exactly one interior interface (the shared `x=0.5` plane), so we expect
`len(face_matches) == 1`.

In [ ]:
face_matches, outer_faces = connectivity(blocks)

print(f"face_matches found: {len(face_matches)}")
print(f"outer_faces found:  {len(outer_faces)}")

Let's look at the raw match. Recall the diagonal convention: block1's `lb`/`ub`
is `[14, 0, 0]` to `[14, 8, 12]` -- axis 0 (`i`) is constant at `14`
(`IMAX-1`, since `nx=15`), so this is block0's **`I=IMAX`** face. Block2's is
`[0, 0, 0]` to `[0, 8, 12]` -- constant `i=0`, block1's **`I=1`** face. That's
exactly the shared `x=0.5` plane, seen from each side. The `orientation`
sub-dict's `permutation_index=-1` / `plane='in-plane'` tells us this is a
plain, non-swapped, non-reversed match (both faces are already numbered in
lock-step) -- unsurprising, since we generated both blocks with identical
nested loops.

In [ ]:
fm0 = dict(face_matches[0])
fm0.pop("match", None)  # internal pandas DataFrame, not part of the exported schema
pp.pprint(fm0)

And the raw `outer_faces` -- ten of them (each block has 6 sides, minus the
1 side each consumed by the interior match above = 5 outer sides x 2 blocks
= 10). **Important**: the `id` field you see here is just a sequential
counter in face-discovery order (`1, 2, 3, ...`) -- it has *no* physical
meaning yet. Every `tag_surfaces_*` function below exists specifically to
overwrite it with a real surface id.

In [ ]:
for f in outer_faces[:3]:
    pp.pprint(f)

## Step 2: Rotational periodicity

Next we peel the rotationally-periodic seam off of `outer_faces` with
`rotated_periodicity`, **before** tagging the remaining outer faces. This
ordering matters (and is what `export_to_plot3d_flatten_deck` itself does
internally): it's what lets the `bc_codes` strategy's "leftover periodic ->
id 6" fallback mean anything (see the format explanation above) -- there's
nothing left over to catch if periodicity hasn't run yet.

Note `rotated_periodicity`'s `rotation_angle` argument is in **degrees**
(unlike `create_rotation_matrix`, used next, which takes **radians** --
easy to mix up, so watch the units).

We expect 2 periodic pairs: each block's own two `theta` faces (`K=1` and
`K=KMAX`) match each other once rotated by `dtheta` -- a self-match within
each block, since every block in this simple mesh spans the full wedge
angle.

In [ ]:
periodic_faces, outer_faces, _, _ = rotated_periodicity(
    blocks, face_matches, outer_faces,
    rotation_angle=math.degrees(dtheta), rotation_axis="x",
)

print(f"periodic_faces found: {len(periodic_faces)}")
print(f"outer_faces remaining: {len(outer_faces)}")

In [ ]:
pf0 = dict(periodic_faces[0])
pf0.pop("match", None)
pp.pprint(pf0)

Block1 here is block0's `K=KMAX` face (`k=12` constant); block2 is the *same
block's* `K=1` face (`k=0` constant) -- confirming the self-match. Only 6
outer faces remain: 1 inlet, 1 outlet, 2 hub, 2 shroud (one hub + one shroud
per block).

### Visualizing block connectivity and periodicity

With `face_matches` (the interior seam between the two blocks) and `periodic_faces` (the rotationally-periodic seams) both computed, let's see *where* they are. Both blocks are drawn as light, mostly transparent context (their full outer envelope) so you can see the overall wedge shape, with the block-to-block interface and the periodic faces highlighted in bold color on top.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D


def _face_coords(block, lb, ub):
    i0, i1 = sorted((lb[0], ub[0]))
    j0, j1 = sorted((lb[1], ub[1]))
    k0, k1 = sorted((lb[2], ub[2]))
    X = block.X[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    Y = block.Y[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    Z = block.Z[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    # squeeze the collapsed (size-1) axis so we get 2D arrays for plot_surface
    return np.squeeze(X), np.squeeze(Y), np.squeeze(Z)


def _block_face_specs(block):
    """lb/ub diagonals for all 6 logical faces (I=1, I=IMAX, ...) of a block."""
    ni, nj, nk = block.IMAX - 1, block.JMAX - 1, block.KMAX - 1
    return [
        ([0, 0, 0], [0, nj, nk]),
        ([ni, 0, 0], [ni, nj, nk]),
        ([0, 0, 0], [ni, 0, nk]),
        ([0, nj, 0], [ni, nj, nk]),
        ([0, 0, 0], [ni, nj, 0]),
        ([0, 0, nk], [ni, nj, nk]),
    ]


fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Light gray context: every block's full outer envelope (all 6 logical faces), low alpha
for block in blocks:
    for lb, ub in _block_face_specs(block):
        try:
            X, Y, Z = _face_coords(block, lb, ub)
            ax.plot_surface(X, Y, Z, color='lightgray', alpha=0.25, shade=False, linewidth=0)
        except Exception as e:
            print(f"warning: skipped context face {lb}-{ub}: {e}")

# Bold magenta: block-to-block interfaces (face_matches), drawn from block1's side
for m in face_matches:
    try:
        b1 = blocks[m['block1']['block_index']]
        X, Y, Z = _face_coords(b1, m['block1']['lb'], m['block1']['ub'])
        ax.plot_surface(X, Y, Z, color='magenta', alpha=0.9, shade=False)
    except Exception as e:
        print(f"warning: skipped face_matches entry: {e}")

# Bold cyan: rotationally-periodic faces, both sides of each pair
if periodic_faces:
    for pf in periodic_faces:
        for side in ('block1', 'block2'):
            try:
                b = blocks[pf[side]['block_index']]
                X, Y, Z = _face_coords(b, pf[side]['lb'], pf[side]['ub'])
                ax.plot_surface(X, Y, Z, color='cyan', alpha=0.9, shade=False)
            except Exception as e:
                print(f"warning: skipped periodic_faces entry ({side}): {e}")
else:
    print("No periodic_faces to draw.")

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Block connectivity')
ax.view_init(elev=15, azim=110)

legend_handles = [
    mpatches.Patch(color='magenta', label='block-to-block interface (face_matches)'),
    mpatches.Patch(color='cyan', label='periodic faces'),
]
ax.legend(handles=legend_handles, loc='upper left')

plt.tight_layout()
plt.show()

The **magenta** patch is the single interior interface from `face_matches` -- the shared `x=0.5` plane where the solver exchanges flux/state directly between block 0 and block 1. This is an **interior face, not a boundary condition**. The two **cyan** patches are the `periodic_faces` pair -- each block's own `K=1`/`K=KMAX` theta faces, related to each other by a rotation of `dtheta` (the blade pitch). Note that here periodicity is a *self*-match within each block, not a match between the two blocks.

## Step 3: Tag the remaining outer faces

### Geometric tagging (the strategy we'll use for the final file)

`tag_surfaces_geometric` looks at each outer face's mean axial position and
radius (computed from the node coordinates, using `axis="x"` as the machine
axis here), compares it against the global axial/radial extent of *all*
outer faces, and classifies: closest to min-x -> **inlet** (id 1), closest to
max-x -> **outlet** (id 2), min-radius -> **hub** (id 4), max-radius ->
**shroud** (id 5), anything else -> **blade** (id 3). We work on a copy so we
can compare strategies side by side below.

In [ ]:
outer_faces_geo = [dict(f) for f in outer_faces]
outer_faces_geo, surface_ids = tag_surfaces_geometric(blocks, outer_faces_geo, axis="x")

print("surface_ids:", surface_ids)
for f in outer_faces_geo:
    pp.pprint(f)

### Visualizing the boundary conditions

Now that every outer face carries a real surface `id` (from `tag_surfaces_geometric`), let's draw each tagged face as a colored 3D surface patch, colored by which physical boundary it is. This is the geometric picture of exactly what `outer_faces_geo` and `surface_ids` describe in text/JSON form above -- and it is the **most important** plot in this tutorial, since it's the thing you should check visually before trusting any `connectivity.json` you generate for a real mesh.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D


def _face_coords(block, lb, ub):
    i0, i1 = sorted((lb[0], ub[0]))
    j0, j1 = sorted((lb[1], ub[1]))
    k0, k1 = sorted((lb[2], ub[2]))
    X = block.X[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    Y = block.Y[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    Z = block.Z[i0:i1 + 1, j0:j1 + 1, k0:k1 + 1]
    # squeeze the collapsed (size-1) axis so we get 2D arrays for plot_surface
    return np.squeeze(X), np.squeeze(Y), np.squeeze(Z)


# Prefer well-known colors for well-known surface names; fall back to
# the default matplotlib color cycle for anything else, so this works
# for any surface_ids map (e.g. one that also has a 'blade' surface).
_preferred_colors = {
    'inlet': 'tab:blue',
    'outlet': 'tab:red',
    'hub': 'tab:green',
    'shroud': 'tab:orange',
    'blade': 'tab:gray',
}
_fallback_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
surface_colors = {}
for idx, (sid, name) in enumerate(surface_ids.items()):
    surface_colors[int(sid)] = _preferred_colors.get(name, _fallback_cycle[idx % len(_fallback_cycle)])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for f in outer_faces_geo:
    try:
        block = blocks[f['block_index']]
        X, Y, Z = _face_coords(block, f['lb'], f['ub'])
        color = surface_colors.get(f['id'], 'black')
        ax.plot_surface(X, Y, Z, color=color, alpha=0.7, shade=False)
    except Exception as e:
        print(f"warning: skipped outer face {f}: {e}")

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Boundary condition surfaces')
ax.view_init(elev=25, azim=-50)

legend_handles = [
    mpatches.Patch(color=surface_colors[int(sid)], label=name)
    for sid, name in surface_ids.items()
]
ax.legend(handles=legend_handles, loc='upper left')

plt.tight_layout()
plt.show()

For this wedge, `surface_ids` (and the colors above) are:

- **tab:blue** = `inlet` (id 1)
- **tab:red** = `outlet` (id 2)
- **tab:gray** = `blade` (id 3) -- present in `surface_ids` by default, but no face is tagged with it here since this simple wedge has no airfoil surface, so this color doesn't appear in the plot above.
- **tab:green** = `hub` (id 4)
- **tab:orange** = `shroud` (id 5)

These are exactly the same integer ids a run YAML's `boundary_conditions:` entries reference in their `surfaces:` list (Step 6 below) -- e.g. `surfaces: [1]` on a `Plot3DFlattenInletBC` targets precisely the blue patch above.

Exactly as expected: the two `j=0` (min radius) faces got `id=4` (hub), the
two `j=JMAX` (max radius) faces got `id=5` (shroud), the min-x face got
`id=1` (inlet), the max-x face got `id=2` (outlet). No face falls outside
those four bands, so no face is tagged `blade` (id 3) here -- our wedge has
no airfoil surface, unlike a real rotor/stator passage.

### Diagonals tagging (explicit, one face at a time)

`tag_surfaces_from_diagonals` is the fully-manual alternative: you specify
the exact `[i,j,k]` corner box for a surface yourself. Useful when geometry
alone can't disambiguate (e.g. two inlets at similar axial stations), or
just to force one specific surface without touching the rest. Here we
re-tag block0's inlet face by its exact diagonal:

In [ ]:
outer_faces_diag = [dict(f) for f in outer_faces]  # fresh copy, untagged
inlet_face = [f for f in outer_faces_diag
              if f["block_index"] == 0 and f["lb"][0] == f["ub"][0] == 0][0]
print("face to tag:", inlet_face)

specs = [{
    "block_index": 0,
    "lb": inlet_face["lb"],
    "ub": inlet_face["ub"],
    "id": 1,
    "name": "inlet",
}]
outer_faces_diag, surface_ids_diag = tag_surfaces_from_diagonals(blocks, outer_faces_diag, specs)
print("surface_ids:", surface_ids_diag)
print("tagged face:", [f for f in outer_faces_diag if f.get("id") is not None])

### BC-codes tagging (from authoritative per-block face codes)

`tag_surfaces_from_bc_codes` takes one 6-integer row per block, in the fixed
`FACE_ORDER = ["I=1", "I=IMAX", "J=1", "J=JMAX", "K=1", "K=KMAX"]`. We hand-build
codes as if this wedge came from a code-tagged legacy mesh. Block 0's `I=1`
is the machine inlet (`5`); its `I=IMAX` is the interior interface (`0`,
already matched, so this code is simply never looked up since that side
isn't in `outer_faces` anymore); `J=1`/`J=JMAX` are hub/shroud (`9`/`109`);
`K=1`/`K=KMAX` are marked periodic (`13`/`14`) -- but since
`rotated_periodicity` already removed those faces from `outer_faces` above,
those two codes also find nothing to tag, exactly as the format explanation
predicted.

In [ ]:
# FACE_ORDER = [I=1, I=IMAX, J=1, J=JMAX, K=1, K=KMAX]
bc_codes = [
    [5, 0, 9, 109, 13, 14],   # block 0: inlet, interface, hub, shroud, periodic, periodic
    [0, 6, 9, 109, 13, 14],   # block 1: interface, outlet, hub, shroud, periodic, periodic
]

outer_faces_bc = [dict(f) for f in outer_faces]  # fresh copy, untagged
outer_faces_bc, surface_ids_bc = tag_surfaces_from_bc_codes(blocks, outer_faces_bc, bc_codes)

print("surface_ids:", surface_ids_bc)
for f in outer_faces_bc:
    pp.pprint(f)

Same result as the geometric pass (ids 1/2/4/5), as expected -- three
different roads to the same answer for this simple case. Now write the
`bc_codes.json` sidecar recording those raw per-block codes:

In [ ]:
bc_codes_path = os.path.join(OUT_DIR, "wedge.bc_codes.json")
bc_codes_payload = write_bc_codes_json(
    blocks, bc_codes, bc_codes_path,
    block_order=["wedge_block_0", "wedge_block_1"],
)
print(json.dumps(bc_codes_payload, indent=2))

## Step 4: The periodicity transformation matrix

`create_rotation_matrix(rotation_angle, rotation_axis)` builds the 3x3
rotation matrix for the `periodicity.transformation_matrix` field --
**radians** this time (this function, unlike `rotated_periodicity` above,
takes radians). We assemble the full `periodicity` metadata dict by hand
here so you see every field `write_connectivity_json` expects; note it's
required whenever `periodic_faces` is non-empty.

In [ ]:
transformation_matrix = create_rotation_matrix(dtheta, "x")
print(transformation_matrix)

In [ ]:
periodicity_meta = {
    "nblades": int(nblades),
    "rotation_axis": "x",
    "rotation_angle_rad": float(dtheta),
    "rotation_angle_deg": float(math.degrees(dtheta)),
    "transformation_matrix": transformation_matrix.tolist(),
    "convention": "Face_B_points = (transformation_matrix @ Face_A_points.T).T",
}
pp.pprint(periodicity_meta)

## Step 5: Write `connectivity.json`

Now we have every ingredient: `face_matches` (1 interior seam),
`outer_faces_geo` (6 faces, geometrically tagged), `periodic_faces` (2
rotational pairs), and `periodicity_meta`. `write_connectivity_json` folds
these into the final payload -- filling in `mesh_file`, `nblocks`, and the
reference `permutation_matrices` block automatically -- and returns the same
dict it wrote, which we'll also reload from disk to prove the round-trip.

In [ ]:
conn_path = os.path.join(OUT_DIR, "wedge.connectivity.json")
payload = write_connectivity_json(
    blocks,
    face_matches,
    outer_faces_geo,
    conn_path,
    periodic_faces=periodic_faces,
    periodicity=periodicity_meta,
    surface_ids=surface_ids,
    mesh_file="wedge.xyz",
)

with open(conn_path) as fh:
    reloaded = json.load(fh)

# NOTE: we deliberately don't `print(json.dumps(reloaded, indent=2))` the
# whole payload here -- `permutation_matrices` alone is a fixed 8-entry
# table of 2x2 matrices that, once pretty-printed, buries every other field
# under a wall of nested brackets. Instead, print each part in the format
# that's actually readable for it: a summary line for the top level, one
# compact (single-line) JSON dict per example record, and the small fixed
# tables printed directly.
print("top-level keys:", list(reloaded.keys()))
print(f"nblocks: {reloaded['nblocks']}")
print(f"face_matches: {len(reloaded['face_matches'])}  "
      f"outer_faces: {len(reloaded['outer_faces'])}  "
      f"periodic_faces: {len(reloaded.get('periodic_faces', []))}")
print("surface_ids:", reloaded["surface_ids"])

print("\nexample face_match:")
print(json.dumps(reloaded["face_matches"][0]))            # compact, one line

print("\nexample outer_face:")
print(json.dumps(reloaded["outer_faces"][0]))              # compact, one line

print("\nperiodicity:")
print(json.dumps(reloaded["periodicity"], indent=2))

print("\npermutation_matrices (8 fixed 2x2 signed permutations):")
for m in reloaded["permutation_matrices"]:
    print("  ", m)                                          # one matrix per line

### Dissecting this file against the schema table

Match this back to the schema table above, field by field:

- **`mesh_file: "wedge.xyz"`**, **`nblocks: 2`** -- as given.
- **`face_matches`** has our 1 interior seam: block0's `I=IMAX`
  (`lb=[14,0,0]`, `ub=[14,8,12]`) matched to block1's `I=1`
  (`lb=[0,0,0]`, `ub=[0,8,12]`), `permutation_index: -1` with the identity
  `permutation_matrix` -- an in-plane, non-reversed match, as expected for
  two blocks generated with identical loops.
- **`outer_faces`** has exactly the 6 tagged faces from Step 3: two `id: 4`
  (hub, one per block), two `id: 5` (shroud, one per block), one `id: 2`
  (outlet, block1's `I=IMAX`), one `id: 1` (inlet, block0's `I=1`). Every
  entry here dropped the transient face-discovery counter and now carries
  only the meaningful surface id.
- **`periodic_faces`** has both self-matches from Step 2 -- block0's
  `K=KMAX` <-> `K=1`, and block1's `K=KMAX` <-> `K=1` -- each still with
  `permutation_index`/`permutation_matrix`, but notice **no `id` field**:
  `write_connectivity_json` strips `id` from match-shaped entries (it only
  belongs on `outer_faces`, where it names a boundary surface -- an interior
  or periodic seam doesn't have one).
- **`periodicity`** carries our `periodicity_meta` dict, plus a `source`
  string `write_connectivity_json` filled in automatically since we didn't
  supply one (it would have used ours if we had).
- **`surface_ids`** is the `{"1": "inlet", ..., "5": "shroud"}` map from the
  geometric tagging pass.
- **`permutation_matrices`** is the fixed reference copy of all 8 canonical
  matrices -- present in every file this exporter writes, regardless of
  which ones any individual match actually used.

## Step 6: Boundary-condition physics

With the topology written, attach physics to surface ids 1 (inlet), 2
(outlet), 4 (hub), 5 (shroud) using the three BC dataclasses. This wedge has
no blade surface (id 3) since we didn't model an airfoil, so we only need
one inlet, one outlet, and two (stationary) walls.

In [ ]:
bcs = [
    Plot3DFlattenInletBC(
        name="wedge_inlet",
        surfaces=[1],
        total_pressure=101325.0,
        total_temperature=288.15,
        turbulence_intensity=0.02,
        turbulence_length_scale=0.001,
        blades_full_ring=nblades,
    ),
    Plot3DFlattenOutletBC(
        name="wedge_outlet",
        surfaces=[2],
        back_pressure=100000.0,
        blades_full_ring=nblades,
    ),
    Plot3DFlattenWallBC(name="wedge_hub", surfaces=[4], thermal="adiabatic", rotating=False),
    Plot3DFlattenWallBC(name="wedge_shroud", surfaces=[5], thermal="adiabatic", rotating=False),
]

bc_yaml_path = os.path.join(OUT_DIR, "wedge_boundary_conditions.yaml")
yaml_text = write_boundary_conditions_yaml(bcs, bc_yaml_path)
print(yaml_text)

Only the fields we actually set appear -- e.g. `Plot3DFlattenWallBC`'s unset
`wall_temperature`/`wall_heat_flux` are simply absent rather than written as
`null`. This YAML fragment is meant to be pasted into (or merged with) the
`boundary_conditions:` list of your plot3d-flatten deck run configuration.

## Step 7: The one-call driver, `export_to_plot3d_flatten_deck`

Everything above -- `connectivity`, `rotated_periodicity`, tagging, writing
the grid, `connectivity.json`, and the BC YAML -- is exactly what
`export_to_plot3d_flatten_deck` does internally in one call. It always uses the
**full** `connectivity()` (never `_fast`, for the reasons discussed above),
and always runs periodicity *before* tagging when a rotation is requested,
matching the order in this tutorial. Now that you've seen each step, this is
the call you'd actually use day to day:

In [ ]:
paths = export_to_plot3d_flatten_deck(
    blocks,
    OUT_DIR,
    "wedge_full",
    rotation_angle=dtheta,      # radians (create_rotation_matrix convention)
    rotation_axis="x",
    nblades=nblades,
    tagging="geometric",
    bcs=bcs,
)

pp.pprint(paths)
print()
print("files in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

`paths` reports only the files actually written: the formatted grid
(`wedge_full.xyz`), `wedge_full.connectivity.json`, and
`wedge_full_boundary_conditions.yaml` (no `bc_codes` key here since we
didn't pass `bc_codes=` to this call -- that key only appears when you do,
or when `tagging="bc_codes"`).

## Step 8: Merging multiple rows

To stand in for a real 2-row machine (e.g. rotor + stator) without building
a second mesh, we merge the *same* wedge connectivity file with itself as
"row 0" and "row 1". In a real case these would be two genuinely different
per-row connectivity files (e.g. `rotor.connectivity.json` +
`stator.connectivity.json`) produced by running this whole tutorial once
per row, and the grids would be concatenated the same way (row 0's blocks,
then row 1's blocks) into a single `.xyz`.

In [ ]:
merged_path = os.path.join(OUT_DIR, "merged.connectivity.json")
row_file = os.path.join(OUT_DIR, "wedge_full.connectivity.json")

merged = merge_connectivity_json([row_file, row_file], merged_path, id_stride=100)

print("merged nblocks:", merged["nblocks"], "(2 blocks/row x 2 rows)")
print("merged surface_ids:", merged["surface_ids"])

`nblocks` doubled (2 -> 4): row 1's blocks are now indices 2-3, not 0-1.
`surface_ids` shows the `id_stride=100` convention in action -- row 0 kept
ids `1..5`, row 1's are offset to `101..105`, exactly like the rotor
(`1..5`)/stator (`101..105`) split in a real multi-row run configuration.
Let's confirm both the block-index and surface-id offsets landed on an
actual row-1 outer face:

In [ ]:
row1_faces = [f for f in merged["outer_faces"] if f["block_index"] >= len(blocks)]
print("a row-1 outer face (block_index offset by 2, id offset by 100):")
pp.pprint(row1_faces[0])

## Step 9: Flatten the mesh into a finite-volume graph

Everything through Step 8 builds the **block-level** graph a GPU solver
consumes as `{case}.xyz` + `{case}.connectivity.json`: which whole block
faces touch which, which faces are exterior boundaries (and what they
physically are), and which faces are rotationally periodic. The solver
itself expands that small description into the actual **per-cell**
finite-volume graph -- the thing a GPU timestep loop walks -- internally,
at load time. Nothing in Steps 1-8 ever touches that per-cell graph from
Python.

`plot3d.flatmesh.flatten_mesh` builds that same kind of per-cell graph
directly in Python, as a plain, solver-agnostic data structure: every
**cell** (hexahedron) becomes a node of the graph, and every **face**
between two cells becomes a directed `owner -> neighbor` edge carrying
real geometry (an area vector, a centroid) instead of just a block-face
corner box. Block-to-block interfaces and periodic pairs are *ordinary
interior edges* of this graph -- the two cells on either side are simply
neighbors; only true physical boundaries (inlet/outlet/hub/shroud/blade)
get a missing (`-1`) neighbor.

This is a genuinely different artifact from `connectivity.json`, not a
reformatting of it. `connectivity.json` is what a GPU solver reads, and
it builds this same finite-volume graph internally from it at load time.
`flatten_mesh` is for **everything else** -- any other CFD solver, a mesh
quality check, custom post-processing, a from-scratch GPU kernel -- that
wants the finite-volume graph directly, in Python, without re-implementing
a solver's own internal mesh builder.

In [ ]:
from plot3d import flatten_mesh, FlatMesh, write_flat_mesh, read_flat_mesh

# Give the hub a nonzero spin so Step 9's "nodes know their BC" demo below
# has an actual rotating_wall to find (every BC in `bcs` up to this point
# modeled a static wedge -- nothing was rotating).
bcs_flat = [
    Plot3DFlattenInletBC(
        name="wedge_inlet",
        surfaces=[1],
        total_pressure=101325.0,
        total_temperature=288.15,
        turbulence_intensity=0.02,
        turbulence_length_scale=0.001,
        blades_full_ring=nblades,
    ),
    Plot3DFlattenOutletBC(
        name="wedge_outlet",
        surfaces=[2],
        back_pressure=100000.0,
        blades_full_ring=nblades,
    ),
    Plot3DFlattenWallBC(
        name="wedge_hub", surfaces=[4], thermal="adiabatic",
        rotating=True, wall_rotation_rate=1256.6,  # rad/s, ~12,000 RPM
    ),
    Plot3DFlattenWallBC(name="wedge_shroud", surfaces=[5], thermal="adiabatic", rotating=False),
]

fmesh = flatten_mesh(
    blocks, face_matches, outer_faces_geo,
    periodic_faces=periodic_faces,
    periodicity=periodicity_meta,
    surface_ids=surface_ids,
    bcs=bcs_flat,
)
print(fmesh.summary())

### The `FlatMesh` structure, field by field

`flatten_mesh` returns a single `FlatMesh` object: plain numpy arrays in
"structure of arrays" form (one array per field, all the same length
within their group) -- easy to hand to any other tool, no graph library
required.

**Points** -- `Nn` welded mesh nodes (block-to-block interface nodes are
merged into one shared point; periodic partner nodes are kept distinct,
since they're rotated copies, not the same point):

- `points` `(Nn, 3)` float64 -- node coordinates.

**Cells** -- `Nc` hexahedra, one per structured `(block, i, j, k)` cell.
This is exactly the `(i,j,k)` picture from the rest of this tutorial:
every cell still remembers which block and which structured cell index it
came from:

- `cell_volume` `(Nc,)` -- cell volume.
- `cell_centroid` `(Nc, 3)` -- cell centroid.
- `cell_vertices` `(Nc, 8)` -- global point ids of the cell's 8 corners
  (`VTK_HEXAHEDRON` order).
- `cell_block_id` `(Nc,)` -- which block (index into `blocks`) this cell
  came from.
- `cell_ijk` `(Nc, 3)` -- that block's structured `(i, j, k)` cell index.

**Faces** -- `Nf` faces, one per cell interface. These are exactly the
I/J/K cell-to-cell interfaces of the structured grid, now carrying real
geometry instead of just an `lb`/`ub` corner box:

- `face_owner` / `face_neighbor` `(Nf,)` -- the two cells (global ids) on
  either side of the face; `face_neighbor == -1` marks a true physical
  boundary (interior, block-to-block, *and* periodic faces all have a
  real neighbor cell).
- `face_area` `(Nf, 3)` -- the face's area *vector* (magnitude = area,
  direction = outward normal from `face_owner`) -- what a finite-volume
  flux gets dotted against.
- `face_normal` `(Nf, 3)` / `face_area_mag` `(Nf,)` -- the same vector
  pre-split into a unit normal and a scalar magnitude, so a consumer
  doesn't have to normalize `face_area` itself.
- `face_centroid` `(Nf, 3)` -- face centroid.
- `face_vertices` `(Nf, 4)` -- global point ids of the face's 4 corners.
- `face_surface_id` `(Nf,)` -- the `surface_ids` id this face belongs to
  (`-1` for interior/periodic faces, which belong to no boundary surface).
- `face_bc_type` `(Nf,)` -- a numeric code, decoded by `bc_type_legend`
  below.
- `face_periodic_rotation` `(Nf,)` -- rotation angle in radians for
  periodic faces (`0` for everything else) -- see the next cell.

**Nodes** -- per-point BC tagging, for solvers that need boundary
conditions on nodes rather than (or in addition to) faces:

- `point_bc_type` `(Nn,)` -- each node's single *dominant* BC type code (a
  node touching more than one boundary, e.g. a hub/blade corner, resolves
  to whichever BC has the highest precedence: rotating wall > wall >
  symmetry > inlet/outlet > periodic > interior).
- `point_surf_off` / `point_surf_ids` -- a CSR (compressed sparse row)
  pair: node `n`'s surface memberships are
  `point_surf_ids[point_surf_off[n]:point_surf_off[n+1]]` (a node can
  belong to more than one surface, e.g. a hub/blade junction node).

**Attributes** (plain dicts, not arrays):

- `surface_ids` -- `{id: name}`, the same map as `connectivity.json`'s.
- `bc_type_legend` -- `{code: name}` decoding `face_bc_type`/`point_bc_type`.
- `boundary_conditions` -- `{surface_id: {...}}`, the resolved physics for
  each surface (from `bcs`), e.g. the hub's `wall_rotation_rate`.
- `meta` -- provenance/counts (`version`, `source`, `nblocks`, `Nc`, `Nf`,
  `Nn`).

In [ ]:
# FlatMesh IS a graph: cells are nodes, faces are directed owner->neighbor
# edges. Pick one plain interior face (via bc_type_legend, so we know it's
# neither periodic nor a boundary) and walk it both ways.
interior_code = [code for code, name in fmesh.bc_type_legend.items()
                 if name == "interior"][0]
interior_faces = np.nonzero(fmesh.face_bc_type == interior_code)[0]
f = int(interior_faces[len(interior_faces) // 2])
c = int(fmesh.face_owner[f])

print(f"cell {c}  (block={int(fmesh.cell_block_id[c])}, ijk={tuple(fmesh.cell_ijk[c].tolist())})")
print(f"  volume   = {fmesh.cell_volume[c]:.6g}")
print(f"  centroid = {fmesh.cell_centroid[c]}")

owner_edges = np.nonzero(fmesh.face_owner == c)[0]
neighbor_edges = np.nonzero(fmesh.face_neighbor == c)[0]
neighbors = sorted(
    (set(fmesh.face_neighbor[owner_edges].tolist()) - {-1})
    | (set(fmesh.face_owner[neighbor_edges].tolist()) - {c})
)
print(f"  neighbor cells = {neighbors}")

print(f"\nface {f}  (an interior face -- cell {c}'s owner side of it):")
print(f"  owner / neighbor = {int(fmesh.face_owner[f])} / {int(fmesh.face_neighbor[f])}")
print(f"  area vector      = {fmesh.face_area[f]}")
print(f"  unit normal      = {fmesh.face_normal[f]}")
print(f"  area magnitude   = {fmesh.face_area_mag[f]:.6g}")

In [ ]:
# Periodic faces are ordinary interior edges too -- their "neighbor" is a
# real cell (here, even in the SAME block, since each block spans the
# full wedge angle), just related by a rotation instead of literal spatial
# adjacency.
periodic_idx = np.nonzero(fmesh.face_periodic_rotation != 0)[0]
pf = int(periodic_idx[0])

print(f"periodic face {pf}:")
print(f"  owner / neighbor = {int(fmesh.face_owner[pf])} / {int(fmesh.face_neighbor[pf])}  "
      f"(neighbor >= 0 -> a real cell, not a boundary)")
print(f"  rotation         = {fmesh.face_periodic_rotation[pf]:.6f} rad   (dtheta = {dtheta:.6f} rad)")
print("Periodic cells are just neighbors in this graph -- the solver rotates "
      "the flux across this face by that angle before adding it to the "
      "neighbor cell's balance.")

In [ ]:
# Nodes carry BC info too: `point_bc_type` (decoded by `bc_type_legend`)
# and the surface-membership CSR `point_surf_off`/`point_surf_ids`.
rotating_code = [code for code, name in fmesh.bc_type_legend.items()
                 if name == "rotating_wall"][0]
node = int(np.nonzero(fmesh.point_bc_type == rotating_code)[0][0])
print(f"node {node} is a {fmesh.bc_type_legend[rotating_code]!r} "
      f"(point_bc_type={rotating_code}) -- the hub we just made rotate")

boundary_node = int(np.nonzero(fmesh.point_surf_off[1:] > fmesh.point_surf_off[:-1])[0][0])
lo, hi = int(fmesh.point_surf_off[boundary_node]), int(fmesh.point_surf_off[boundary_node + 1])
member_ids = fmesh.point_surf_ids[lo:hi].tolist()
print(f"node {boundary_node} belongs to surface id(s) {member_ids} "
      f"({[fmesh.surface_ids[s] for s in member_ids]})")

print("\nboundary_conditions (resolved physics per surface id):")
pp.pprint(fmesh.boundary_conditions)

In [ ]:
h5_path = os.path.join(OUT_DIR, "wedge.h5")
npz_path = os.path.join(OUT_DIR, "wedge.npz")
vtu_path = os.path.join(OUT_DIR, "wedge.vtu")

try:
    fmesh.to_hdf5(h5_path)
    print("wrote", h5_path)
except ImportError as e:
    print(
        f"skipped {h5_path}: {e}\n"
        'Install the optional HDF5 extra with:  pip install "plot3d[hdf5]"'
    )

fmesh.to_npz(npz_path)
print("wrote", npz_path)

fmesh.to_vtu(vtu_path)
print("wrote", vtu_path)

print("\nfiles in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

# write_flat_mesh/read_flat_mesh are the top-level, one-call convenience
# wrappers around flatten_mesh + FlatMesh.to_*/from_*: write_flat_mesh(...)
# builds the graph AND writes it in one call (dispatching on the path's
# extension), and read_flat_mesh(path) reads it back. Round-trip through
# .npz (no optional dependency) to confirm it's the same graph:
roundtrip_path = os.path.join(OUT_DIR, "wedge_roundtrip.npz")
write_flat_mesh(
    blocks, face_matches, outer_faces_geo, roundtrip_path,
    periodic_faces=periodic_faces, periodicity=periodicity_meta,
    surface_ids=surface_ids, bcs=bcs_flat,
)
fmesh_reloaded = read_flat_mesh(roundtrip_path)
print(f"round-tripped: Nc={fmesh_reloaded.cell_volume.shape[0]} "
      f"(original Nc={fmesh.cell_volume.shape[0]})")

### `.h5` vs `.npz` vs `.vtu` -- no proprietary format

All three writers use standard, widely-supported containers -- nothing
proprietary, and nothing that requires this package to read it back:

- **`.h5`** (HDF5, via `to_hdf5`/`from_hdf5`) -- the richest layout (named
  groups/datasets/attrs, as laid out above), readable from Python
  (`h5py`), Rust (the `hdf5` crate), C/C++/Fortran (the official HDF5
  libraries), MATLAB, and more. Requires the optional `hdf5` extra:
  `pip install "plot3d[hdf5]"` (plain `pip install plot3d` does not pull
  in `h5py` -- that's exactly what the `try`/`except ImportError` above
  guards against).
- **`.npz`** (via `to_npz`/`from_npz`) -- pure numpy, zero optional
  dependencies (works out of the box with plain `plot3d`), and readable
  outside Python too (e.g. Rust's `npyz` crate).
- **`.vtu`** (via `to_vtu`, write-only) -- a hand-written VTK XML
  `UnstructuredGrid` file with no `vtk`/`meshio` dependency at all; opens
  directly in ParaView for visual inspection of cell volumes, block ids,
  and node BC types.

Pick `.npz` for a dependency-free default, `.h5` when you want the fully
self-describing layout (or need it from Rust/C/Fortran), and `.vtu` purely
for looking at the mesh in ParaView.

## How a GPU CFD solver uses these files

Finally, here's how a run deck ties everything together -- `mesh:` points
at the grid + connectivity.json this tutorial produces, and
`boundary_conditions:` is exactly the YAML fragment from Step 6 (or the
merged, offset ids from Step 8 for a multi-row case), pasted straight in.
Excerpted from an example plot3d-flatten deck run configuration:

```yaml
mesh:
  grid_file: "../mesh/stage.xyz"
  connectivity_file: "../mesh/stage.connectivity.json"
  format: formatted

# Surface-id map (id-stride 100):
#   rotor  (blocks 0-5):        1 inlet, 2 outlet(mixing-plane), 3 blade, 4 hub, 5 shroud
#   stator (blocks 6-10, +100): 101 inlet(mixing-plane), 102 machine outlet, 103 blade, 104 hub, 105 shroud
boundary_conditions:
  - name: m_inlet
    type: inlet
    surfaces: [1]
    total_pressure: 101325.0
    total_temperature: 288.17
    blades_full_ring: 36

  - name: rotor35_blade
    type: wall
    surfaces: [3]
    thermal: adiabatic
    rotating: true

  - name: stator35_shroud
    type: wall
    surfaces: [105]
    thermal: adiabatic
```

At load time, the GPU solver reads the `.xyz` grid, reads
`connectivity.json` to learn the block-level graph (which faces touch,
which are exterior and what they are, which are periodic), **builds its own
internal flat per-cell dual graph** from those two pieces, and then applies
each `boundary_conditions:` entry to every cell face whose parent surface
`id` is in that entry's `surfaces:` list. Nothing in this Python package
ever touches a per-cell graph -- everything we produced here is the small,
human-readable block-level description the solver expands from.

## Cleanup

In [ ]:
import shutil
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
print("Removed", OUT_DIR)